# Datamine INTEXT Process Example

This notebook demonstrates how to configure and run the **`intext`** process wrapper in `dmstudio`.

## Process Description

## INTEXT

Import a text file with a known data definition to create a Datamine binary file (either .dm or .dmx, depending on your current **[default file format](<../COMMON/Datamine-File-Format.md>)**.

A data definition can either be determined from the header of the imported file, or from an existing Datamine binary file. If a separate file (**INDD**) is used, **INTEXT** attempts to calculate the type and (in the case of alphanumerics) size of the columns of the output file.

Several parameters are available to control the conversion process.

**INTEXT** is used by the [Text Importer](<../COMMON/text-importer.md>) tool, where many of the settings listed below appear as curated screen controls.

Datamine data is generated in the current default [Datamine File Formats](<../COMMON/Datamine-File-Format.md>).

* **Note** (Data records may not start with the character "!". This is because the ! symbol acts as the end-of-data character. However, macro files, where the command starts with !, may be read if a blank is inserted prior to the ! symbol.):

### Input Files:

* **indd** (Table):
  File containing Data Definition. Ignored if SETTINGS is specified.
  Required=No

* **settings** (Table):
  File providing settings for each imported data column, with up to four optional fields.
  See "SETTINGS and INDD inputs", above. If both INDD and SETTINGS are specified, INDD is
  ignored.
  Required=No

### Output Files:

* **out** (Table):
  File to be created.
  Required=Yes

### Fields:

### Parameters:

* **column**:
  Define the column separator in the input text file.
  Range=
  Values=
  Default=
  Required=No

* **delimit**:
  Treat consecutive delimiters as one.
  Range=
  Values=
  Default=
  Required=No

* **fixwidth**:
  Import data as fixed width columns. If **FIXWIDTH** =1 then **SETTINGS** must exist and
  contain a list of column widths for all importable fields (**COLWIDTH** attribute). If
  **FIXWIDTH** = 0, the column width is variable, and set according to the data in the
  imported file.
  Range=
  Values=
  Default=
  Required=No

* **startrow**:
  Row number in input text file of the first record to import.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **endrow**:
  Row number in input text file of the last record to import. Default=+, i.e. read all
  rows. 0 or negative will also be treated as read all rows.
  Range=Undefined
  Values=Undefined
  Default="+"
  Required=No

* **header**:
  Row number in input text file that contains the field names. If no header is set, then
  the input data defintion (**INDD**) is required (see also **ALLCOL**). 0 or negative is
  treated as no header.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **upcase**:
  Force all field names to be upper case. Note that if the data definition or input file
  contains multiple variations of a field name case, and **UPCASE** = 1, the import will
  stop.
  Range=
  Values=
  Default=
  Required=No

* **decimal**:
  Specify whether numbers use decimal points or commas.
  Range=
  Values=
  Default=
  Required=No

* **quotestr**:
  Single character used to quote strings in the imported data. The value can either be an
  integer code or a single character. Integer codes: 0 no quotes 1 double quotes (") 2
  single quote (') Any column separator (according to COLUMN) found inside a quoted string
  is considered a normal text character. Note: Multiline strings are not supported.
  Range=
  Values=
  Default=
  Required=No

* **comment**:
  Single character used to identify comment lines in the incoming file. Any line starting
  with this character is ignored. The value can either be an integer code or single
  character. Integer codes: 0 no comment 1 # character
  Range=
  Values=
  Default=
  Required=No

* **incrmnt**:
  Only read every 1 of **INCRMNT** records / rows, starting from **STARTROW**.
  Range=Undefined
  Values=Undefined
  Default=1
  Required=No

* **absent**:
  Character string or number that defines absent data.
  Range=Undefined
  Values=Undefined
  Default="-"
  Required=No

* **tracedat**:
  A value that represents a trace numeric value. If this value is detected, it is
  assigned the system "TR" flag in the numeric field of the output file.
  Range=
  Values=
  Default=
  Required=No

* **ceiling**:
  Character string or number that defines ceiling data.
  Range=Undefined
  Values=Undefined
  Default="+"
  Required=No

* **allcol**:
  Only used if **INDD** is specified. Import all columns even if these do not exist in
  Data Definition.
  Range=
  Values=
  Default=
  Required=No

* **nscan**:
  Number of records to read to auto detect field types and lengths. Not used if **IN** is
  defined and **HEADER** is 0 or negative. If 0, let the process determine the appropriate
  number of rows to scan according to the file size.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **worldxyz**:
  This parameter is only used when importing block model. It is mainly designed for
  rotated block model when the incoming coordinates are not local grid coordinates as
  expected, but world (or mine local) coordinates. In this case use the value 1. =1 : if
  you know that the XC,YC,ZC come as world coordinates. =0 : if you know that the XC,YC,ZC
  come as grid local coordinates. =-1 : let the process choose (local grid coordinates for
  rotated block models, world coordinates otherwise) (default).
  Range=-1,1
  Values=-1,0,1
  Default=-1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('intext')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute intext
print("Running intext...")
dm_cmd.intext(
    indd_i='optional',  # required
    settings_i='optional',  # required
    out_o='t_intext_out',  # required
    # column_p='optional',  # optional
    # delimit_p='optional',  # optional
    # fixwidth_p='optional',  # optional
    # startrow_p=1,  # optional
    # endrow_p='+',  # optional
    # header_p=1,  # optional
    # upcase_p='optional',  # optional
    # decimal_p='optional',  # optional
    # quotestr_p='optional',  # optional
    # comment_p='optional',  # optional
    # incrmnt_p=1,  # optional
    # absent_p='-',  # optional
    # tracedat_p='optional',  # optional
    # ceiling_p='+',  # optional
    # allcol_p='optional',  # optional
    # nscan_p=0,  # optional
    # worldxyz_p=-1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("intext execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_intext_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")